# AMBER protein MD setup

The dashboard provides `input.pdb`; this notebook prepares AMBER inputs through equilibration with `pmemd`, and writes the production control file.
Edit the parameter cell for `name`, production length, equilibration lengths, salt concentration, and solvent buffer.

In [ ]:
name = "protein"  # give me a better name

nanoseconds = 0.05  # just for fun
nsteps = int(nanoseconds * 500000)  # assumes 2 fs timestep

nmin = 5000  # minimization cycles per stage
nheat = 25000  # heating steps (25 ps at 1 fs)
nnvt = 10000  # NVT equilibration steps (10 ps at 1 fs)
nnpt = 12500  # NPT equilibration steps (25 ps at 2 fs)
temp0 = 300.0  # target temperature (K)
heat_start_temp = 0.0  # initial heating temperature (K)
salt_mM = 150.0  # salt concentration
buffer = 9.0  # solvation buffer (angstroms)

In [ ]:
import contextlib
import re
import statistics

import amber_wrapper as amb
import matplotlib.pyplot as plt
import nglview as nv

## Inspect Input Chemistry

Before setup, inspect chains, ligands, protonation-sensitive groups, missing residues or atoms, and disulfides.
Fix chemistry issues upstream; LEaP cannot infer all biological intent from coordinates.

In [ ]:
nv.show_file("input.pdb")

## Prepare PDB

`pdb4amber` standardizes the structure for LEaP.
Inspect its messages for residue renaming, disulfides, alternate locations, missing atoms, and nonstandard chemistry.

In [ ]:
amb.pdb4amber(i="input.pdb", o=f"{name}_clean.pdb", reduce=True)

## Build Topology And Solvate

LEaP builds an ff19SB/OPC topology, solvates the system, neutralizes charge, and adds salt.
Edit the force field, water model, box geometry, buffer, or ion policy in this cell if the system requires different choices.

In [ ]:
with open(f"{name}.leap.in", "w") as leap:
    leap.write(f"""\
source leaprc.protein.ff19SB
source leaprc.water.opc
protein = loadpdb {name}_clean.pdb
check protein
saveamberparm protein {name}_gas.prmtop {name}_gas.rst7
addions protein Na+ 0
addions protein Cl- 0
solvateoct protein OPCBOX {buffer}
quit
""")

# Salt calculation: run tleap once to get water count,
# then write a final script with salt and saveamberparm.
amb.tleap(f=f"{name}.leap.in")

# Parse leap.log for number of water residues
with open("leap.log") as log:
    content = log.read()
match = re.search(r"Added\s+(\d+)\s+residues", content)
if match:
    n_wat = int(match.group(1))
    n_ion_pairs = int(n_wat * salt_mM / 56000)
    print(f"Water molecules: {n_wat}, adding {n_ion_pairs} ion pairs")
else:
    n_ion_pairs = 0
    print("Could not detect water count; skipping salt buffer")

# Build final leap script in one f-string.
salt_cmd = f"addionsrand protein Na+ {n_ion_pairs} Cl- {n_ion_pairs}\n" if n_ion_pairs > 0 else ""
with open(f"{name}.leap.in", "w") as leap:
    leap.write(f"""\
source leaprc.protein.ff19SB
source leaprc.water.opc
protein = loadpdb {name}_clean.pdb
check protein
saveamberparm protein {name}_gas.prmtop {name}_gas.rst7
addions protein Na+ 0
addions protein Cl- 0
solvateoct protein OPCBOX {buffer}
{salt_cmd}saveamberparm protein {name}_solv.prmtop {name}_solv.rst7
savepdb protein {name}_solv.pdb
quit
""")

amb.tleap(f=f"{name}.leap.in")

In [ ]:
nv.show_file(f"{name}_solv.pdb")

## Minimize 1: Restrained Solvent And Ions

Relax solvent and ions while restraining solute heavy atoms.
This catches bad contacts without allowing the protein structure to drift immediately.

In [ ]:
with open("min1.mdin", "w") as m:
    m.write(f"""Restrained solvent and ion minimization
&cntrl
  imin=1, ncyc={nmin // 2}, maxcyc={nmin}, ntmin=1,
  ntb=1, cut=10.0,
  ntr=1,
  restraint_wt=50.0,
  restraintmask='!:WAT,Cl-,Na+ & !@H=',
  ntpr=50,
/
""")

amb.pmemd(
    i="min1.mdin",
    o="min1.mdout",
    p=f"{name}_solv.prmtop",
    c=f"{name}_solv.rst7",
    r="min1.rst7",
    ref=f"{name}_solv.rst7",
    O=True,
)

## Minimize 2: Unrestrained System

Remove restraints and relax the full system.
Check `min2.mdout` for convergence and warnings before heating.

In [ ]:
with open("min2.mdin", "w") as m:
    m.write(f"""Unrestrained minimization
&cntrl
  imin=1, ncyc={nmin // 2}, maxcyc={nmin}, ntmin=1,
  ntb=1, cut=10.0,
  ntr=0,
  ntpr=50,
/
""")

amb.pmemd(i="min2.mdin", o="min2.mdout", p=f"{name}_solv.prmtop", c="min1.rst7", r="min2.rst7", O=True)

In [ ]:
# Convert restart to PDB for visualization
amb.ambpdb(p=f"{name}_solv.prmtop", c="min2.rst7", o="min2.pdb")
nv.show_file("min2.pdb")

## Heat: Restrained NVT

Ramp from 0 K to `temp0` under NVT while restraining solute heavy atoms.
Inspect the temperature trace for a smooth rise before continuing.

In [ ]:
with open("heat.mdin", "w") as h:
    h.write(f"""Restrained NVT heating
&cntrl
  imin=0, irest=0, ntx=1,
  ntb=1, cut=10.0,
  ntc=2, ntf=2,
  ntt=3, gamma_ln=1.0, ig=-1,
  tempi={heat_start_temp}, temp0={temp0}, nmropt=1,
  ntr=1,
  restraint_wt=10.0,
  restraintmask='!:WAT,Cl-,Na+ & !@H=',
  nstlim={nheat}, dt=0.001,
  ntpr=100, ntwx=1000, ntwr=5000,
  ioutfm=1, iwrap=1, ntxo=2,
/
&wt TYPE='TEMP0', istep1=0, istep2={nheat}, value1={heat_start_temp}, value2={temp0}
/
&wt TYPE='END'
/
""")

amb.pmemd(
    i="heat.mdin",
    o="heat.mdout",
    p=f"{name}_solv.prmtop",
    c="min2.rst7",
    r="heat.rst7",
    ref="min2.rst7",
    O=True,
)

In [ ]:
# Extract temperature from mdout and plot.
steps, temp = [], []
in_results = False
with open("heat.mdout") as mdout:
    for line in mdout:
        if not in_results and (line.startswith(" NSTEP") or line.startswith(" NSTEP =")):
            in_results = True
        if not in_results:
            continue
        if "A V E R A G E S" in line:
            break
        if "TEMP(K)" in line:
            parts = line.split()
            with contextlib.suppress(ValueError, IndexError):
                nstep = int(parts[2])
                t = float(parts[8])
                if not steps or nstep > steps[-1]:
                    steps.append(nstep)
                    temp.append(t)

if temp:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, temp)
    plt.title("Heating phase temperature")
    plt.xlabel("Step")
    plt.ylabel("Temperature (K)")
    plt.grid()
    plt.show()

## NVT Equilibration: Restrained

Equilibrate at constant volume with solute heavy-atom restraints.
The temperature should fluctuate around `temp0` without sustained drift.

In [ ]:
with open("nvt.mdin", "w") as nvt:
    nvt.write(f"""Restrained NVT equilibration
&cntrl
  imin=0, irest=1, ntx=5,
  ntb=1, cut=10.0,
  ntc=2, ntf=2,
  ntt=3, gamma_ln=1.0, ig=-1, temp0={temp0},
  ntr=1,
  restraint_wt=5.0,
  restraintmask='!:WAT,Cl-,Na+ & !@H=',
  nstlim={nnvt}, dt=0.001,
  ntpr=100, ntwx=1000, ntwr=5000,
  ioutfm=1, iwrap=1, ntxo=2,
/
""")

amb.pmemd(
    i="nvt.mdin", o="nvt.mdout", p=f"{name}_solv.prmtop", c="heat.rst7", r="nvt.rst7", ref="heat.rst7", O=True
)

In [ ]:
# Plot temperature from NVT equilibration.
steps, temp = [], []
in_results = False
with open("nvt.mdout") as mdout:
    for line in mdout:
        if not in_results and (line.startswith(" NSTEP") or line.startswith(" NSTEP =")):
            in_results = True
        if not in_results:
            continue
        if "A V E R A G E S" in line:
            break
        if "TEMP(K)" in line:
            parts = line.split()
            with contextlib.suppress(ValueError, IndexError):
                nstep = int(parts[2])
                t = float(parts[8])
                if not steps or nstep > steps[-1]:
                    steps.append(nstep)
                    temp.append(t)

if temp:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, temp)
    plt.title("NVT equilibration temperature")
    plt.xlabel("Step")
    plt.ylabel("Temperature (K)")
    plt.grid()
    plt.show()
    mean_t = sum(temp) / len(temp)
    std_t = statistics.stdev(temp) if len(temp) > 1 else 0.0
    print(f"Mean T = {mean_t:.2f} K,  std = {std_t:.2f} K")

## NPT Equilibration: Restrained

Continue with weaker solute heavy-atom restraints while equilibrating at constant pressure.
Monitor density and pressure before using `npt.rst7` for production.

In [ ]:
with open("npt.mdin", "w") as npt:
    npt.write(f"""Restrained NPT equilibration
&cntrl
  imin=0, irest=1, ntx=5,
  ntb=2, cut=10.0,
  ntc=2, ntf=2,
  ntt=3, gamma_ln=1.0, ig=-1, temp0={temp0},
  ntp=1, barostat=1, pres0=1.0, taup=2.0,
  ntr=1,
  restraint_wt=2.5,
  restraintmask='!:WAT,Cl-,Na+ & !@H=',
  nstlim={nnpt}, dt=0.002,
  ntpr=100, ntwx=1000, ntwr=5000,
  ioutfm=1, iwrap=1, ntxo=2,
/
""")

amb.pmemd(i="npt.mdin", o="npt.mdout", p=f"{name}_solv.prmtop", c="nvt.rst7", r="npt.rst7", ref="nvt.rst7", O=True)

In [ ]:
# Plot temperature, pressure, and density from NPT equilibration.
data = {"Step": [], "Press": [], "Density": [], "Temp": []}
current = None
in_results = False
with open("npt.mdout") as mdout:
    for line in mdout:
        if not in_results and (line.startswith(" NSTEP") or line.startswith(" NSTEP =")):
            in_results = True
        if not in_results:
            continue
        if "A V E R A G E S" in line:
            break
        parts = line.split()
        if "TEMP(K)" in line:
            with contextlib.suppress(ValueError, IndexError):
                nstep = int(parts[2])
                t = float(parts[8])
                if not data["Step"] or nstep > data["Step"][-1]:
                    data["Step"].append(nstep)
                    data["Temp"].append(t)
                    data["Press"].append(None)
                    data["Density"].append(None)
                    current = len(data["Step"]) - 1
        if current is None:
            continue
        if "PRESS" in line or "PRES" in line:
            for i, part in enumerate(parts):
                if part in ("PRESS", "PRES"):
                    with contextlib.suppress(ValueError, IndexError):
                        data["Press"][current] = float(parts[i + 2])
                    break
        if "DENSITY" in line or "DENSTY" in line or "Density" in line:
            for i, part in enumerate(parts):
                if part in ("DENSITY", "DENSTY", "Density"):
                    with contextlib.suppress(ValueError, IndexError):
                        data["Density"][current] = float(parts[i + 2])
                    break

fig, axes = plt.subplots(3, 1, figsize=(10, 10))
for ax, key in zip(axes, ("Press", "Density", "Temp")):
    points = [(step, val) for step, val in zip(data["Step"], data[key]) if val is not None]
    if points:
        ax.plot([step for step, _ in points], [val for _, val in points])
    ax.set_ylabel(key)
    ax.grid()
axes[-1].set_xlabel("Step")
plt.suptitle("NPT equilibration")
plt.tight_layout()
plt.show()

## Production Handoff

This notebook writes `prod.mdin` and stops before production.
Run production with `prod.mdin`, `{name}_solv.prmtop`, and `npt.rst7`; the Dashboard/runtime may replace `pmemd` with another AMBER engine.

In [ ]:
with open("prod.mdin", "w") as prod:
    prod.write(f"""Unrestrained NPT production
&cntrl
  imin=0, irest=1, ntx=5,
  ntb=2, cut=10.0,
  ntc=2, ntf=2,
  ntt=3, gamma_ln=1.0, ig=-1, temp0={temp0},
  ntp=1, barostat=2, pres0=1.0,
  ntr=0,
  nstlim={nsteps}, dt=0.002,
  ntpr=5000, ntwx=5000, ntwr=5000,
  ioutfm=1, iwrap=1, ntxo=2,
/
""")

print("Wrote prod.mdin. Production is not run in this notebook.")
print(f"pmemd -O -i prod.mdin -o prod.mdout -p {name}_solv.prmtop -c npt.rst7 -r prod.rst7 -x prod.nc")